# 01. Explore Data
Load data, basic stats, weather exploration.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

sys.path.append(os.path.join(os.path.dirname(os.path.abspath('.')), 'src'))
import utils
import const

# Load Data
base_df = utils.load_data()
print(f"Total records: {len(base_df):,}")
print(f"Date range: {base_df['start_date'].min()} to {base_df['start_date'].max()}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(base_df['arrival_delay'], bins=100) 
plt.title('Distribution of Arrival Delays (Raw Data)')
plt.xlabel('Delay (seconds)')
plt.ylabel('Count')
plt.yscale('log')
plt.show()

In [ ]:
base_df['arrival_delay'].describe(percentiles=[0.01, 0.05, 0.95, 0.99]).round(2)

In [ ]:
per001 = base_df['arrival_delay'].quantile(0.01)
per099 = base_df['arrival_delay'].quantile(0.99)

In [ ]:
# Filter outliers
base_df_filtered = base_df[(base_df['arrival_delay'] >= per001) & (base_df['arrival_delay'] <= per099)]
print(f"Records before filtering outliers: {len(base_df):,}")
del base_df
base_df_filtered = base_df_filtered[base_df_filtered['alert_effect_detour'] >= 0]
print(f"Records after filtering outliers: {len(base_df_filtered):,}")

In [ ]:
base_df_filtered.isnull().sum()

In [ ]:
# identify stop_ids with null region_id
null_region_df = base_df_filtered.loc[base_df_filtered['region_id'].isnull()]
null_region_df[['stop_id']].drop_duplicates()

In [ ]:
stop_times_path = const.STOP_TIMES_TXT
stop_times_df = pd.read_csv(stop_times_path, usecols=['trip_id', 'stop_id', 'stop_sequence'])
stop_times_df['trip_id'] = stop_times_df['trip_id'].astype(str)

target_date = '20251121'
target_route = base_df_filtered.loc[base_df_filtered['start_date'] == target_date, 'route_id'].mode().iloc[0]

target_df = base_df_filtered[
    (base_df_filtered['route_id'] == target_route) & 
    (base_df_filtered['start_date'] == target_date)
]

stop_route_counts = base_df_filtered.groupby('stop_id')['route_id'].nunique()
most_recorded_trip_id = target_df['trip_id'].value_counts().idxmax()
trip_data = target_df[target_df['trip_id'] == most_recorded_trip_id].sort_values('stop_sequence').copy()
trip_data['routes_at_stop'] = trip_data['stop_id'].map(stop_route_counts)

# Create visualization
fig, ax1 = plt.subplots(figsize=(14, 6))

# Plot 1: Arrival Delay (Left Axis)
color1 = 'tab:blue'
ax1.set_xlabel('Stop Sequence')
ax1.set_ylabel('Arrival Delay (seconds)', color=color1, fontsize=12)
ax1.plot(trip_data['stop_sequence'], trip_data['arrival_delay'], 
         color=color1, marker='o', linestyle='-', linewidth=2, label='Arrival Delay')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.grid(True, alpha=0.3)

# Plot 2: Number of Routes at Stop (Right Axis)
ax2 = ax1.twinx()
color2 = 'tab:orange'
ax2.set_ylabel('Number of Routes at Stop', color=color2, fontsize=12)
ax2.bar(trip_data['stop_sequence'], trip_data['routes_at_stop'], 
        color=color2, alpha=0.3, width=0.4, label='Routes Count')
ax2.tick_params(axis='y', labelcolor=color2)

plt.title(f"Delay Evolution vs Stop Connectivity (Trip: {most_recorded_trip_id})")
fig.legend(loc="upper left", bbox_to_anchor=(0.1, 0.9))
plt.tight_layout()
plt.show()

Since I retrieved data in 2 min interval, there are the situation that specific stops have more than 2 arrival delays or no delay data. I should adjust each trip had one delay. 

## Timeline Metrix
### 1 hour interval

In [ ]:
base_df_filtered.groupby('hour_of_day')['arrival_delay'].mean().plot(kind='bar', figsize=(10, 5))
plt.title('Average Arrival Delay by Hour of Day')
plt.xticks(rotation=0)
plt.xlabel('Hour of Day')
plt.ylabel('Average Delay (seconds)')
plt.show()

### 10 min interval

In [ ]:
base_df_filtered['minute'] = base_df_filtered['actual_arrival_time'].dt.minute
base_df_filtered['minute_bin'] = (base_df_filtered['minute'] // 10) * 10

delay_by_10min = base_df_filtered.groupby(['hour_of_day', 'minute_bin'])['arrival_delay'].mean().reset_index()
delay_by_10min['time_axis'] = delay_by_10min['hour_of_day'] + delay_by_10min['minute_bin'] / 60

plt.figure(figsize=(15, 6))
plt.plot(delay_by_10min['time_axis'], delay_by_10min['arrival_delay'], marker='.', linestyle='-')
plt.title('Average Arrival Delay by 10-Minute Interval')
plt.xlabel('Hour of Day')
plt.ylabel('Average Delay (seconds)')
plt.xticks(range(0, 25))
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

There are two peaks in a day. One of them is placed around seven. It's outstanding higher than other times at seven, which is considered that morning commution focuses at seven. Second peak occurs around at 5pm. It is more gentle slope than former one. It might indicate that returing time is splitted compred to coming time.

In [ ]:
# Map day_of_week to names (ISODOW: 1=Monday, 7=Sunday)
day_map = {1: 'Monday', 2: 'Tuesday', 3: 'Wednesday', 4: 'Thursday', 5: 'Friday', 6: 'Saturday', 7: 'Sunday'}
daily_delays = base_df_filtered.groupby('day_of_week')['arrival_delay'].mean()
daily_delays.index = daily_delays.index.map(day_map)

daily_delays.plot(kind='bar', figsize=(10, 5))
plt.title('Average Arrival Delay by Day of Week')
plt.xticks(rotation=0)
plt.xlabel('Day of Week')
plt.ylabel('Average Delay (seconds)')
plt.show()

There is the highest value on Friday. We can refer that many people go out on weekend. Also, it's really low on Monday. Most of workers in hybrid company don't commute on Monday.

In [ ]:
plt.figure(figsize=(12, 6))
pivot_data = base_df_filtered.pivot_table(index='hour_of_day', columns='day_of_week', values='arrival_delay', aggfunc='mean')
pivot_data.columns = pivot_data.columns.map(day_map)

sns.heatmap(pivot_data, cmap='coolwarm', center=0, annot=False)
plt.title('Average Arrival Delay by Hour and Day')
plt.show()

This graph is more understandable. There is a peak acound seven on middle of week. It shows irregler commutions. Also, coming weekend, it increses to go out at night.

## Region Metrix

In [ ]:
trip_df = pd.read_csv(const.TRIPS_TXT)
route_df = pd.read_csv(const.ROUTES_TXT)
stop_df = pd.read_csv(const.STOPS_TXT)

# filter data with bus
bus_route = route_df[route_df['route_type'] == 3]['route_id']
bus_trip = trip_df.merge(bus_route, on=['route_id'], how='inner')
route_names = bus_trip[['route_id', 'direction_id', 'trip_headsign', 'trip_id']].groupby(['route_id', 'direction_id', 'trip_headsign']).agg({'trip_id': 'size', 'trip_id': 'first'}).reset_index()
# route_names.rename(columns={'trip_id': 'trip_count'}, inplace=True)
route_names.head()

In [ ]:
# Create label
route_names['label'] = route_names.apply(lambda x: f"{x['trip_headsign']}", axis=1)

# Plot
plot_data = route_names.sort_values('trip_count', ascending=False).head(10)

plt.figure(figsize=(12, 6))
plt.bar(plot_data['label'], plot_data['trip_count'], color='skyblue')
plt.xlabel('Route (Headsign)')
plt.ylabel('Number of Trips')
plt.xticks(rotation=45, ha='right')
plt.title('Top 10 Routes by Number of Trips')
plt.show()

In [ ]:
# Calculate mean delay
top_delays = base_df_filtered.groupby(['region_id', 'route_id', 'direction_id']).agg({'arrival_delay': 'mean'}).reset_index()
top_delays.rename(columns={'trip_id': 'trip_count'}, inplace=True)

# Ensure route_id is string in both
top_delays['route_id'] = top_delays['route_id'].astype(str)
route_names['route_id'] = route_names['route_id'].astype(str)

# Merge
top_delays = top_delays.merge(route_names, on=['route_id', 'direction_id'], how='left')

# Create label
top_delays['label'] = top_delays.apply(lambda x: f"{x['trip_headsign']}\n{x['region_id']}", axis=1)

# Plot
plot_data = top_delays.sort_values('arrival_delay', ascending=False).head(8)

plt.figure(figsize=(12, 6))
plt.bar(plot_data['label'], plot_data['arrival_delay'], color='skyblue')
plt.xlabel('Route (Headsign)')
plt.ylabel('Average Delay (s)')
plt.xticks(rotation=45, ha='right')
plt.title('Top 10 Routes by Average Arrival Delay')
plt.show()

It shows the top 10 average delays of routes. Half of them are in Richmond. 

In [ ]:
# Calculate mean delay in Vancouver area
top_delays_van = top_delays[top_delays['region_id'] == 'vancouver']

# Plot
plot_data = top_delays_van.sort_values('arrival_delay', ascending=False).head(10)

plt.figure(figsize=(12, 6))
plt.bar(plot_data['label'], plot_data['arrival_delay'], color='skyblue')
plt.xlabel('Route (Headsign)')
plt.ylabel('Average Delay (s)')
plt.xticks(rotation=45, ha='right')
plt.title('Top 10 Routes by Average Arrival Delay in Vancouver Area')
plt.show()

In [ ]:
import folium
from folium.plugins import TimestampedGeoJson
import numpy as np
import pandas as pd

# Calculate coordinates
stop_hourly_stats['stop_lat'] = np.degrees(np.arctan2(stop_hourly_stats['lat_sin'], stop_hourly_stats['lat_cos']))
stop_hourly_stats['stop_lon'] = np.degrees(np.arctan2(stop_hourly_stats['lon_sin'], stop_hourly_stats['lon_cos']))

# Split into start and end stops
start_stops = stop_hourly_stats[stop_hourly_stats['stop_sequence'] == stop_hourly_stats['min_stop_sequence']].copy()
end_stops = stop_hourly_stats[stop_hourly_stats['stop_sequence'] == stop_hourly_stats['max_stop_sequence']].copy()

# Rename columns
start_stops = start_stops.rename(columns={'stop_lat': 'start_lat', 'stop_lon': 'start_lon', 'arrival_delay': 'start_delay'})
end_stops = end_stops.rename(columns={'stop_lat': 'end_lat', 'stop_lon': 'end_lon', 'arrival_delay': 'end_delay'})

# Merge start and end stops
route_lines = pd.merge(
    start_stops[['route_id', 'direction_id', 'hour_of_day', 'start_lat', 'start_lon']],
    end_stops[['route_id', 'direction_id', 'hour_of_day', 'end_lat', 'end_lon', 'end_delay']],
    on=['route_id', 'direction_id', 'hour_of_day'],
    how='inner'
)

# 1. Select Target Routes (Fixed Top 5 per Region based on Total Trip Count)
region_route_counts = base_df_filtered.groupby(['region_id', 'route_id'])['trip_id'].nunique().reset_index(name='trip_count')
top_routes = region_route_counts.sort_values(['region_id', 'trip_count'], ascending=[True, False]).groupby('region_id').head(5)
target_routes = top_routes['route_id'].unique()

print(f"Selected Fixed Routes ({len(target_routes)} unique routes): {target_routes}")

# 2. Filter the hourly data for these fixed routes
route_lines = route_lines[route_lines['route_id'].isin(target_routes)]

features = []
min_delay = route_lines['end_delay'].min()
max_delay = route_lines['end_delay'].max()

for _, row in route_lines.iterrows():
    # Normalize delay for weight (thickness)
    if max_delay > min_delay:
        normalized_delay = (row['end_delay'] - min_delay) / (max_delay - min_delay)
        weight = 2 + 8 * normalized_delay  # Scale weight between 2 and 10
    else:
        weight = 5
    
    # Color based on direction
    try:
        dir_val = int(row['direction_id'])
    except:
        dir_val = -1
        
    if dir_val == 0:
        color = 'blue'
    elif dir_val == 1:
        color = 'red'
    else:
        color = 'green'
        
    # Calculate arrow wings with corrected aspect ratio
    arrow_wings = get_arrow_wings(
        row['start_lat'], row['start_lon'], 
        row['end_lat'], row['end_lon']
    )
    
    # Main line + arrow wings
    coordinates = [
        [[row['start_lon'], row['start_lat']], [row['end_lon'], row['end_lat']]]
    ] + arrow_wings

    feature = {
        'type': 'Feature',
        'geometry': {
            'type': 'MultiLineString',
            'coordinates': coordinates
        },
        'properties': {
            'time': f"2023-01-01T{int(row['hour_of_day']):02d}:00:00",
            'style': {
                'color': color,
                'weight': weight,
                'opacity': 0.7
            },
            'popup': f"Time: {int(row['hour_of_day']):02d}:00, Route: {row['route_id']}, Dir: {row['direction_id']}, Delay: {row['end_delay']:.1f}s"
        }
    }
    features.append(feature)

m_lines = folium.Map(location=[49.2827, -123.1207], zoom_start=11)

TimestampedGeoJson(
    {'type': 'FeatureCollection', 'features': features},
    period='PT1H',
    duration='PT1H',
    add_last_point=False,
    auto_play=False,
    loop=True,
    max_speed=1,
    loop_button=True,
    date_options='HH:mm'
).add_to(m_lines)

# Add Legend
legend_html = '''
     <div style="position: fixed; 
     bottom: 50px; left: 50px; width: 130px; height: 80px; 
     border:2px solid grey; z-index:9999; font-size:14px;
     background-color:white; opacity: 0.8;
     padding: 10px">
     <b>Direction</b><br>
     <i style="background:blue; width:10px; height:10px; display:inline-block;"></i> Direction 0<br>
     <i style="background:red; width:10px; height:10px; display:inline-block;"></i> Direction 1<br>
     </div>
     '''
m_lines.get_root().html.add_child(folium.Element(legend_html))

m_lines

In [ ]:
base_df_filtered.groupby('region_id')['arrival_delay'].mean().sort_values(ascending=False).plot(kind='bar', figsize=(10, 5))
plt.title('Average Arrival Delay by Region')
plt.xticks(rotation=45, ha='right')
plt.xlabel('Region ID')
plt.ylabel('Average Delay (seconds)')
plt.show()

In [ ]:
region_order = base_df_filtered.groupby('region_id')['arrival_delay'].mean().sort_values(ascending=False).index
pivot_data = base_df_filtered.pivot_table(index='day_of_week', columns='region_id', values='arrival_delay', aggfunc='mean')
pivot_data.index = pivot_data.index.map(day_map)
pivot_data = pivot_data.reindex(columns=region_order)

sns.heatmap(pivot_data, cmap='coolwarm', center=0, annot=False)
plt.title('Average Arrival Delay by Hour and Region')
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
pivot_data = base_df_filtered.pivot_table(index='hour_of_day', columns='region_id', values='arrival_delay', aggfunc='mean')

# Reorder columns based on region_order
pivot_data = pivot_data.reindex(columns=region_order)

sns.heatmap(pivot_data, cmap='coolwarm', center=0, annot=False)
plt.title('Average Arrival Delay by Hour and Region')
plt.show()

In [ ]:
import geopandas as gpd
from shapely import wkt
import folium
from folium.plugins import HeatMap
import numpy as np
# Load regions data
regions_df = pd.read_csv(const.REGIONS_CSV)

# Clean WKT (remove SRID prefix if present)
regions_df['boundary_clean'] = regions_df['boundary'].apply(lambda x: x.split(';')[-1] if isinstance(x, str) and ';' in x else x)

# Convert to GeoDataFrame
# Filter out rows where boundary is null or invalid if necessary
regions_df = regions_df.dropna(subset=['boundary_clean'])
regions_df['geometry'] = regions_df['boundary_clean'].apply(wkt.loads)
gdf_regions = gpd.GeoDataFrame(regions_df, geometry='geometry')

# Set CRS to EPSG:4326 (WGS 84) as indicated by SRID=4326 in the CSV
gdf_regions.set_crs(epsg=4326, inplace=True)

# Calculate mean delay by region
region_delays = base_df_filtered.groupby('region_id')['arrival_delay'].mean().reset_index()

# Merge
gdf_regions = gdf_regions.merge(region_delays, on='region_id', how='left')

# Fill NaN delays with 0 or handle them (e.g., for visualization purposes)
gdf_regions['arrival_delay'] = gdf_regions['arrival_delay'].fillna(0)

# Create Map
m_regions = folium.Map(location=[49.2827, -123.1207], zoom_start=10)

# Add Choropleth
folium.Choropleth(
    geo_data=gdf_regions,
    name='choropleth',
    data=gdf_regions,
    columns=['region_id', 'arrival_delay'],
    key_on='feature.properties.region_id',
    fill_color='YlOrRd',
    fill_opacity=0.6,
    line_opacity=0.2,
    legend_name='Average Arrival Delay (s)'
).add_to(m_regions)

# Add tooltips
folium.GeoJson(
    gdf_regions,
    tooltip=folium.GeoJsonTooltip(fields=['region_name', 'arrival_delay'], aliases=['Region:', 'Avg Delay:']),
    style_function=lambda x: {'color': 'transparent', 'fillColor': 'transparent', 'weight': 0}
).add_to(m_regions)

folium.LayerControl().add_to(m_regions)

# Add stops in no region
stops_no_region = null_region_df.groupby(['stop_id', 'lat_sin', 'lat_cos', 'lon_sin', 'lon_cos']).agg({'arrival_delay': 'mean'}).reset_index()
# Recover Lat/Lon (in degrees)
stops_no_region['stop_lat'] = np.degrees(np.arctan2(stops_no_region['lat_sin'], stops_no_region['lat_cos']))
stops_no_region['stop_lon'] = np.degrees(np.arctan2(stops_no_region['lon_sin'], stops_no_region['lon_cos']))

# Add markers for stops not in any region to make them stand out
for _, row in stops_no_region.iterrows():
    folium.CircleMarker(
        location=[row['stop_lat'], row['stop_lon']],
        radius=5,
        color='blue',
        fill=True,
        fill_color='blue',
        fill_opacity=1.0, # Make it fully opaque to stand out
        popup=f"Stop ID: {row['stop_id']}\nDelay: {row['arrival_delay']:.2f}s",
        tooltip="No Region Stop (Outlier)"
    ).add_to(m_regions)

m_regions

In [ ]:
# 地域境界データを使った遅延の地図可視化
import geopandas as gpd
import folium
from folium.plugins import HeatMap
from shapely import wkt
import pandas as pd
import numpy as np

# ローカル地域境界データの読み込み
gdf_boundary = gpd.read_file(const.BOUNDARY_GEOJSON)

# GeoJSONから読み込んだ時点でGeoDataFrameだが、CRSを確認/設定
if gdf_boundary.crs is None:
    gdf_boundary.set_crs(epsg=4326, inplace=True)

# base_df_filteredからユニークなstop_idと座標を取得
# lat_sin, lat_cos, lon_sin, lon_cos から緯度経度を復元
unique_stops = base_df_filtered[['stop_id', 'lat_sin', 'lat_cos', 'lon_sin', 'lon_cos']].drop_duplicates()
unique_stops['stop_lat'] = np.degrees(np.arctan2(unique_stops['lat_sin'], unique_stops['lat_cos']))
unique_stops['stop_lon'] = np.degrees(np.arctan2(unique_stops['lon_sin'], unique_stops['lon_cos']))

# 座標をGeoDataFrameに変換
gdf_stops = gpd.GeoDataFrame(
    unique_stops,
    geometry=gpd.points_from_xy(unique_stops.stop_lon, unique_stops.stop_lat),
    crs="EPSG:4326"
)

# 空間結合 (Spatial Join) して各バス停がどの地域に含まれるか判定
# gdf_boundaryの 'name' 列が地域名
gdf_joined = gpd.sjoin(gdf_stops, gdf_boundary[['name', 'geometry']], how="left", predicate="within")

# stop_id と 地域名の対応辞書作成
stop_to_area = gdf_joined.set_index('stop_id')['name'].to_dict()

# base_df_filtered に地域名を追加
base_df_filtered['local_area_name'] = base_df_filtered['stop_id'].map(stop_to_area)

# 地域ごとの平均遅延を算出
area_delays = base_df_filtered.groupby('local_area_name')['arrival_delay'].mean().reset_index()
area_delays.rename(columns={'local_area_name': 'name'}, inplace=True) # GeoJSONの列名に合わせる

# 地域境界データに遅延情報をマージ
# 'name' 列で結合
gdf_boundary_merged = gdf_boundary.merge(area_delays, on='name', how='left')
gdf_boundary_merged['arrival_delay'] = gdf_boundary_merged['arrival_delay'].fillna(0)

# 地図作成（中心はバンクーバー）
m_local_area = folium.Map(location=[49.25, -123.12], zoom_start=11)

# Choroplethレイヤ追加
folium.Choropleth(
    geo_data=gdf_boundary_merged,
    name='choropleth',
    data=gdf_boundary_merged,
    columns=['name', 'arrival_delay'],
    key_on='feature.properties.name',
    fill_color='YlOrRd',
    fill_opacity=0.6,
    line_opacity=0.2,
    legend_name='Average Arrival Delay (s)'
).add_to(m_local_area)

# ツールチップ追加
folium.GeoJson(
    gdf_boundary_merged,
    tooltip=folium.GeoJsonTooltip(fields=['name', 'arrival_delay'], aliases=['Local Area:', 'Avg Delay:']),
    style_function=lambda x: {'color': 'transparent', 'fillColor': 'transparent', 'weight': 0}
).add_to(m_local_area)

folium.LayerControl().add_to(m_local_area)

m_local_area

In [ ]:
from folium.plugins import HeatMapWithTime
stop_hourly_stats = base_df_filtered.groupby(['stop_id', 'stop_sequence', 'hour_of_day']).agg({
    'arrival_delay': 'mean',
    'lat_sin': 'first',
    'lat_cos': 'first',
    'lon_sin': 'first',
    'lon_cos': 'first'
}).reset_index()
stop_hourly_stats['stop_lat'] = np.degrees(np.arctan2(stop_hourly_stats['lat_sin'], stop_hourly_stats['lat_cos']))
stop_hourly_stats['stop_lon'] = np.degrees(np.arctan2(stop_hourly_stats['lon_sin'], stop_hourly_stats['lon_cos']))
delay_min = stop_hourly_stats['arrival_delay'].min()
delay_max = stop_hourly_stats['arrival_delay'].max()
# Normalize delay to [0, 1] for heatmap intensity
stop_hourly_stats['delay_weight'] = (stop_hourly_stats['arrival_delay'] - delay_min) / (delay_max - delay_min)


hourly_heat_data = []
hour_labels = []
for hour in range(24):
    hour_subset = stop_hourly_stats[stop_hourly_stats['hour_of_day'] == hour]
    heat_points = hour_subset[['stop_lat', 'stop_lon', 'delay_weight']].values.tolist()
    hourly_heat_data.append(heat_points)
    hour_labels.append(f"{hour:02d}:00")

vancouver_stop_map_timelapse = folium.Map(location=[49.2827, -123.1207], zoom_start=11)
HeatMapWithTime(
    data=hourly_heat_data,
    index=hour_labels,
    radius=10,
    auto_play=False,
    name='Hourly Delay Heatmap'
).add_to(vancouver_stop_map_timelapse)
folium.LayerControl().add_to(vancouver_stop_map_timelapse)

# vancouver_stop_map
vancouver_stop_map_timelapse

In [ ]:
test = base_df_filtered.groupby(['route_id', 'direction_id']).agg({
    'stop_sequence': ['min','max']
})
test.columns = ['min_stop_sequence', 'max_stop_sequence']
test = test.reset_index()
stop_hourly_stats = base_df_filtered.groupby(['route_id', 'direction_id', 'stop_sequence', 'hour_of_day']).agg({
    'arrival_delay': 'mean',
    'lat_sin': 'first',
    'lat_cos': 'first',
    'lon_sin': 'first',
    'lon_cos': 'first'
}).reset_index()
stop_hourly_stats = pd.merge(stop_hourly_stats, test, on=['route_id', 'direction_id'], how='inner')
stop_hourly_stats = stop_hourly_stats[
    (stop_hourly_stats['stop_sequence'] == stop_hourly_stats['min_stop_sequence']) | 
    (stop_hourly_stats['stop_sequence'] == stop_hourly_stats['max_stop_sequence'])
]

In [ ]:
import folium
from folium.plugins import TimestampedGeoJson
import numpy as np
import pandas as pd
import math

def get_arrow_wings(start_lat, start_lon, end_lat, end_lon, scale=0.1):
    # Aspect ratio correction for Vancouver (~49 deg N)
    # This ensures the arrow looks like a proper arrow on the map projection
    lat_rad = math.radians((start_lat + end_lat) / 2)
    aspect = math.cos(lat_rad)
    
    dx = (end_lon - start_lon) * aspect
    dy = end_lat - start_lat
    length = math.sqrt(dx*dx + dy*dy)
    if length == 0:
        return []
    
    # Normalized direction vector
    ux = dx / length
    uy = dy / length
    
    # Arrow wings - 30 degrees from the line
    angle = math.radians(30)
    cos_a = math.cos(angle)
    sin_a = math.sin(angle)
    
    # Calculate backward vector (-ux, -uy) rotated by angle
    # Wing 1
    w1x = (-ux) * cos_a - (-uy) * sin_a
    w1y = (-ux) * sin_a + (-uy) * cos_a
    
    # Wing 2 (rotate other way)
    w2x = (-ux) * cos_a - (-uy) * (-sin_a)
    w2y = (-ux) * (-sin_a) + (-uy) * cos_a
    
    # Scale and unproject longitude
    p1 = [end_lon + (w1x * scale) / aspect, end_lat + (w1y * scale)]
    p2 = [end_lon + (w2x * scale) / aspect, end_lat + (w2y * scale)]
    
    return [
        [[end_lon, end_lat], p1],
        [[end_lon, end_lat], p2]
    ]

# Calculate coordinates
stop_hourly_stats['stop_lat'] = np.degrees(np.arctan2(stop_hourly_stats['lat_sin'], stop_hourly_stats['lat_cos']))
stop_hourly_stats['stop_lon'] = np.degrees(np.arctan2(stop_hourly_stats['lon_sin'], stop_hourly_stats['lon_cos']))

# Split into start and end stops
start_stops = stop_hourly_stats[stop_hourly_stats['stop_sequence'] == stop_hourly_stats['min_stop_sequence']].copy()
end_stops = stop_hourly_stats[stop_hourly_stats['stop_sequence'] == stop_hourly_stats['max_stop_sequence']].copy()

# Rename columns
start_stops = start_stops.rename(columns={'stop_lat': 'start_lat', 'stop_lon': 'start_lon', 'arrival_delay': 'start_delay'})
end_stops = end_stops.rename(columns={'stop_lat': 'end_lat', 'stop_lon': 'end_lon', 'arrival_delay': 'end_delay'})

# Merge start and end stops
route_lines = pd.merge(
    start_stops[['route_id', 'direction_id', 'hour_of_day', 'start_lat', 'start_lon']],
    end_stops[['route_id', 'direction_id', 'hour_of_day', 'end_lat', 'end_lon', 'end_delay']],
    on=['route_id', 'direction_id', 'hour_of_day'],
    how='inner'
)

# 1. Select Target Routes (Fixed Top 5 per Region based on Total Trip Count)
region_route_counts = base_df_filtered.groupby(['region_id', 'route_id'])['trip_id'].nunique().reset_index(name='trip_count')
top_routes = region_route_counts.sort_values(['region_id', 'trip_count'], ascending=[True, False]).groupby('region_id').head(5)
target_routes = top_routes['route_id'].unique()

print(f"Selected Fixed Routes ({len(target_routes)} unique routes): {target_routes}")

# 2. Filter the hourly data for these fixed routes
route_lines = route_lines[route_lines['route_id'].isin(target_routes)]

features = []
min_delay = route_lines['end_delay'].min()
max_delay = route_lines['end_delay'].max()

for _, row in route_lines.iterrows():
    # Normalize delay for weight (thickness)
    if max_delay > min_delay:
        normalized_delay = (row['end_delay'] - min_delay) / (max_delay - min_delay)
        weight = 2 + 8 * normalized_delay  # Scale weight between 2 and 10
    else:
        weight = 5
    
    # Color based on direction
    try:
        dir_val = int(row['direction_id'])
    except:
        dir_val = -1
        
    if dir_val == 0:
        color = 'blue'
    elif dir_val == 1:
        color = 'red'
    else:
        color = 'green'
        
    # Calculate arrow wings with corrected aspect ratio
    arrow_wings = get_arrow_wings(
        row['start_lat'], row['start_lon'], 
        row['end_lat'], row['end_lon']
    )
    
    # Main line + arrow wings
    coordinates = [
        [[row['start_lon'], row['start_lat']], [row['end_lon'], row['end_lat']]]
    ] + arrow_wings

    feature = {
        'type': 'Feature',
        'geometry': {
            'type': 'MultiLineString',
            'coordinates': coordinates
        },
        'properties': {
            'time': f"2023-01-01T{int(row['hour_of_day']):02d}:00:00",
            'style': {
                'color': color,
                'weight': weight,
                'opacity': 0.7
            },
            'popup': f"Time: {int(row['hour_of_day']):02d}:00, Route: {row['route_id']}, Dir: {row['direction_id']}, Delay: {row['end_delay']:.1f}s"
        }
    }
    features.append(feature)

m_lines = folium.Map(location=[49.2827, -123.1207], zoom_start=11)

TimestampedGeoJson(
    {'type': 'FeatureCollection', 'features': features},
    period='PT1H',
    duration='PT1H',
    add_last_point=False,
    auto_play=False,
    loop=True,
    max_speed=1,
    loop_button=True,
    date_options='HH:mm'
).add_to(m_lines)

# Add Legend
legend_html = '''
     <div style="position: fixed; 
     bottom: 50px; left: 50px; width: 130px; height: 80px; 
     border:2px solid grey; z-index:9999; font-size:14px;
     background-color:white; opacity: 0.8;
     padding: 10px">
     <b>Direction</b><br>
     <i style="background:blue; width:10px; height:10px; display:inline-block;"></i> Direction 0<br>
     <i style="background:red; width:10px; height:10px; display:inline-block;"></i> Direction 1<br>
     </div>
     '''
m_lines.get_root().html.add_child(folium.Element(legend_html))

m_lines

## Alert Metrix

In [ ]:
# Improved Approach: Distribution Comparison & Controlled Aggregation
# Strict trip_id matching discards too much data. 
# Instead, we compare distributions and group by (Route, Hour) to control for context.

import scipy.stats as stats

# 1. Overall Distribution (Box Plot)
plt.figure(figsize=(10, 6))
sns.boxenplot(x='has_active_alert', y='arrival_delay', data=base_df_filtered)
plt.title('Distribution of Arrival Delays: Alert vs No Alert')
plt.xticks([0, 1], ['No Alert', 'With Alert'])
plt.ylabel('Arrival Delay (s)')
plt.show()

# Statistical Test (Mann-Whitney U test due to non-normal distribution likely)
# Check if we have both True and False values
if base_df_filtered['has_active_alert'].nunique() > 1:
    alert_delays = base_df_filtered[base_df_filtered['has_active_alert'] == True]['arrival_delay']
    no_alert_delays = base_df_filtered[base_df_filtered['has_active_alert'] == False]['arrival_delay']

    u_stat, p_val = stats.mannwhitneyu(alert_delays, no_alert_delays, alternative='two-sided')
    print(f"Mann-Whitney U Test: p-value = {p_val:.4e} (Significant difference if < 0.05)")
    print(f"Mean Delay -> Alert: {alert_delays.mean():.2f}s, No Alert: {no_alert_delays.mean():.2f}s")
else:
    print("Data does not contain both Alert and No-Alert samples.")

# 2. Controlled Comparison: Group by Route & Hour
# This compares alerts vs no-alerts happening on the same route at the same time of day (across different days)
# This is more robust than trip_id matching because it allows comparing different trips on the same schedule pattern.
grouped_comparison = base_df_filtered.groupby(['route_id', 'hour_of_day', 'has_active_alert'])['arrival_delay'].mean().unstack()

# Proceed only if we have both columns (True/False or 0/1 depending on data type)
if grouped_comparison.shape[1] == 2:
    # Rename for clarity (assuming False/0 is first, True/1 is second)
    col_names = {False: 'No Alert', True: 'With Alert', 0: 'No Alert', 1: 'With Alert'}
    grouped_comparison = grouped_comparison.rename(columns=col_names)
    
    # Ensure we have the target columns
    if 'No Alert' in grouped_comparison.columns and 'With Alert' in grouped_comparison.columns:
        # Filter groups that have data for both conditions
        valid_groups = grouped_comparison.dropna()

        print(f"\nNumber of (Route, Hour) buckets with both conditions: {len(valid_groups)}")

        if len(valid_groups) > 0:
            # Plotting the effect: "With Alert" - "No Alert"
            valid_groups['Delay Impact'] = valid_groups['With Alert'] - valid_groups['No Alert']
            
            plt.figure(figsize=(12, 6))
            sns.histplot(valid_groups['Delay Impact'], kde=True, bins=50)
            plt.axvline(x=0, color='r', linestyle='--', label='No Impact')
            plt.axvline(x=valid_groups['Delay Impact'].mean(), color='g', linestyle='-', label=f"Mean Impact: {valid_groups['Delay Impact'].mean():.1f}s")
            plt.title('Distribution of Alert Impact on Delay (per Route & Hour)')
            plt.xlabel('Delay Difference (With Alert - No Alert) [s]')
            plt.legend()
            plt.show()
            
            # Show top impacted scenarios
            print("\nTop 5 Scenarios (Route, Hour) where Alert Increases Delay:")
            print(valid_groups.sort_values('Delay Impact', ascending=False)['Delay Impact'].head())
        else:
            print("Not enough overlap in Route/Hour buckets.")
    else:
        print("Column renaming failed or columns missing.")
else:
    print("Grouping did not result in two columns (Alert/No Alert). Check data availability.")

In [ ]:
# Analyze Impact of "Near Train Station" Stops
# ==========================================

from sklearn.neighbors import BallTree
import numpy as np

# A. Identify Rail/Train Stops
# ----------------------------
# 1. Load GTFS files
gtfs_dir = const.GTFS_STATIC_DIR
routes_df = pd.read_csv(os.path.join(gtfs_dir, 'routes.txt'))
trips_df = pd.read_csv(const.TRIPS_TXT)
stop_times_df_full = pd.read_csv(const.STOP_TIMES_TXT, usecols=['trip_id', 'stop_id'])
stop_times_df_full['stop_id'] = stop_times_df_full['stop_id'].astype(str)
stops_df = pd.read_csv(const.STOPS_TXT, usecols=['stop_id', 'stop_name', 'stop_lat', 'stop_lon'])
stops_df['stop_id'] = stops_df['stop_id'].astype(str)

# 2. Filter for Rail Routes (Type 0=Tram, 1=Subway, 2=Rail)
#    Usually buses are type 3.
rail_routes = routes_df[routes_df['route_type'].isin([0, 1, 2])]
rail_route_ids = rail_routes['route_id'].unique()

print(f"Found {len(rail_route_ids)} rail routes.")

# 3. Get Rail Trips and Stops
rail_trips = trips_df[trips_df['route_id'].isin(rail_route_ids)]['trip_id'].unique()
rail_stats_stop_ids = stop_times_df_full[stop_times_df_full['trip_id'].isin(rail_trips)]['stop_id'].unique()
rail_stops = stops_df[stops_df['stop_id'].isin(rail_stats_stop_ids)].copy()

print(f"Found {len(rail_stops)} rail station stops (platforms/entrances).")

# B. Find ALL Bus Stops within 500m of each Rail Stop
# ---------------------------------------------------
# 1. Get Lat/Lon of all unique Bus Stops in our analyis dataset
bus_stop_ids_in_data = base_df_filtered['stop_id'].unique()

bus_stops = stops_df[stops_df['stop_id'].isin(bus_stop_ids_in_data)].copy()
print(f"Matched {len(bus_stops)} bus stops from GTFS data.")

if len(bus_stops) == 0:
    print("Error: No bus stops matched. Check IDs.")
else:
    # 2. Use BallTree for query_radius search
    #    Convert lat/lon to radians for BallTree (Haversine metric)
    bus_stops_rad = np.deg2rad(bus_stops[['stop_lat', 'stop_lon']].values)
    rail_stops_rad = np.deg2rad(rail_stops[['stop_lat', 'stop_lon']].values)
    
    # Haversine metric requires (lat, lon) in radians
    # BallTree expects sample as [lat, lon] if metric='haversine'
    tree = BallTree(rail_stops_rad, metric='haversine')
    
    # 500 meters radius
    # Earth radius ~ 6371 km = 6371000 m
    radius_meters = 500
    radius_rad = radius_meters / 6371000
    
    # Query for each bus stop: is any rail stop within radius?
    # query_radius returns an array of indices. If array is not empty, it's near.
    counts = tree.query_radius(bus_stops_rad, r=radius_rad, count_only=True)
    
    # Flag
    bus_stops['is_near_station'] = counts > 0
    
    # Create the set of "near station" stop_ids
    station_connected_stop_ids = set(bus_stops[bus_stops['is_near_station']]['stop_id'].unique())
    
    print(f" identified {len(station_connected_stop_ids)} bus stops near rail stations (<= {radius_meters}m).")

In [ ]:
# ==========================================
# Statistical Analysis: Station Proximity vs Delay
# ==========================================

# 1. Apply the station flag to the entire dataset
# Ensure stop_id is string
base_df_filtered['stop_id'] = base_df_filtered['stop_id'].astype(str)

# Create binary flag
base_df_filtered['is_near_station'] = base_df_filtered['stop_id'].isin(station_connected_stop_ids)

# 2. Calculate Statistics
station_stats = base_df_filtered.groupby('is_near_station')['arrival_delay'].agg(['mean', 'median', 'std', 'count']).reset_index()
station_stats['Label'] = station_stats['is_near_station'].map({True: 'Near Station', False: 'Regular Stop'})

print("--- Delay Statistics by Station Proximity ---")
print(station_stats[['Label', 'mean', 'median', 'std', 'count']])

# 3. Visualize
plt.figure(figsize=(12, 6))

# Bar plot for Means
plt.subplot(1, 2, 1)
sns.barplot(x='Label', y='mean', data=station_stats, palette=['skyblue', 'salmon'])
plt.title('Average Arrival Delay')
plt.ylabel('Average Delay (seconds)')
plt.xlabel('')

# Boxen plot for Distribution (better for large datasets than boxplot)
plt.subplot(1, 2, 2)
# Sample data for plotting if dataset is too large, to speed up visualization
sample_size = min(100000, len(base_df_filtered))
sns.boxenplot(x='is_near_station', y='arrival_delay', data=base_df_filtered.sample(sample_size), palette=['skyblue', 'salmon'])
plt.xticks([0, 1], ['Regular Stop', 'Near Station'])
plt.title(f'Delay Distribution (Sample n={sample_size})')
plt.xlabel('')
plt.ylabel('Delay (seconds)')

plt.tight_layout()
plt.show()

# 4. Difference
mean_near = station_stats.loc[station_stats['is_near_station'] == True, 'mean'].values[0]
mean_regular = station_stats.loc[station_stats['is_near_station'] == False, 'mean'].values[0]
mean_diff = mean_near - mean_regular

print(f"\nAverage Delay Increase at Station Stops: {mean_diff:.2f} seconds")
print(f"  - Regular Stops avg: {mean_regular:.2f} s")
print(f"  - Near Station avg:  {mean_near:.2f} s")

In [ ]:
# ==========================================
# Deep Dive: Station Stops - Trip Start vs En-Route
# ==========================================

# We want to see if delays at stations are due to the bus originating there (waiting for departure)
# or passing through (delayed arrival).

# 1. Flag Trip Starts
# Assumption: stop_sequence 1 is the start. 
base_df_filtered['is_trip_start'] = base_df_filtered['stop_sequence'] == 1

# 2. Filter: Only look at stops Near Stations
station_subset = base_df_filtered[base_df_filtered['is_near_station']].copy()

print(f"Analyzing {len(station_subset)} records at stops near stations...")

# Check if we have data for both conditions
if station_subset['is_trip_start'].nunique() > 1:
    # 3. Compare Start vs En-Route
    trip_start_stats = station_subset.groupby('is_trip_start')['arrival_delay'].agg(['mean', 'median', 'std', 'count']).reset_index()
    trip_start_stats['Condition'] = trip_start_stats['is_trip_start'].map({True: 'Trip Start (Origin)', False: 'En-Route (Passing/Terminating)'})

    print("\n--- Delay at Stations: Trip Start vs En-Route ---")
    print(trip_start_stats[['Condition', 'mean', 'median', 'std', 'count']])

    # 4. Visualization
    plt.figure(figsize=(14, 6))

    # A. Impact at Stations
    plt.subplot(1, 2, 1)
    sns.barplot(x='Condition', y='mean', data=trip_start_stats, palette='viridis')
    plt.title('Average Delay at Stations: Is it the Trip Start?')
    plt.ylabel('Average Delay (s)')
    plt.xlabel('')

    # B. Interaction Heatmap: Station Proximity x Trip Start (All Data)
    # To see the broad picture
    plt.subplot(1, 2, 2)
    pivot_interaction = base_df_filtered.pivot_table(
        index='is_near_station', 
        columns='is_trip_start', 
        values='arrival_delay', 
        aggfunc='mean'
    )
    # Rename for display
    pivot_interaction.index = pivot_interaction.index.map({True: 'Near Station', False: 'Regular Stop'})
    pivot_interaction.columns = pivot_interaction.columns.map({True: 'Trip Start', False: 'Mid-Trip'})

    sns.heatmap(pivot_interaction, annot=True, fmt=".1f", cmap="YlGnBu", cbar_kws={'label': 'Avg Delay (s)'})
    plt.title('Avg Delay: Station Proximity vs Trip Phase')

    plt.tight_layout()
    plt.show()
else:
    print("Insufficient data variability in 'is_trip_start' for station stops.")

In [ ]:
# ==========================================
# Statistical Analysis: Multi-Route Hubs vs Delay
# ==========================================

# 1. Reuse stop_route_counts from previous analysis
# If variable is lost, recalculate
if 'stop_route_counts' not in locals():
    stop_route_counts = base_df_filtered.groupby('stop_id')['route_id'].nunique()

# 2. Add flag to base dataframe
# map returns NaN if stop_id not found, fill with 1 (conservative)
base_df_filtered['route_count_at_stop'] = base_df_filtered['stop_id'].map(stop_route_counts).fillna(1)
base_df_filtered['is_multi_route'] = base_df_filtered['route_count_at_stop'] > 1

# 3. Calculate Statistics
hub_stats = base_df_filtered.groupby('is_multi_route')['arrival_delay'].agg(['mean', 'median', 'std', 'count']).reset_index()
hub_stats['Label'] = hub_stats['is_multi_route'].map({True: 'Multi-Route Hub', False: 'Single-Route Stop'})

print("--- Delay Statistics by Stop Connectivity ---")
print(hub_stats[['Label', 'mean', 'median', 'std', 'count']])

# 4. Visualize
plt.figure(figsize=(12, 6))

# Bar plot for Means
plt.subplot(1, 2, 1)
sns.barplot(x='Label', y='mean', data=hub_stats, palette=['lightgreen', 'orange'])
plt.title('Average Arrival Delay: Single vs Multi Route')
plt.ylabel('Average Delay (seconds)')
plt.xlabel('')

# Boxen plot for Distribution
plt.subplot(1, 2, 2)
sample_size = min(100000, len(base_df_filtered))
sns.boxenplot(x='is_multi_route', y='arrival_delay', data=base_df_filtered.sample(sample_size), palette=['lightgreen', 'orange'])
plt.xticks([0, 1], ['Single-Route', 'Multi-Route'])
plt.title(f'Delay Distribution (Sample n={sample_size})')
plt.xlabel('')
plt.ylabel('Delay (seconds)')

plt.tight_layout()
plt.show()

# 5. Difference
if len(hub_stats) == 2:
    mean_hub = hub_stats.loc[hub_stats['is_multi_route'] == True, 'mean'].values[0]
    mean_single = hub_stats.loc[hub_stats['is_multi_route'] == False, 'mean'].values[0]
    diff_hub = mean_hub - mean_single

    print(f"\nAverage Delay Increase at Multi-Route Hubs: {diff_hub:.2f} seconds")
    print(f"  - Single-Route Stops avg: {mean_single:.2f} s")
    print(f"  - Multi-Route Hubs avg:   {mean_hub:.2f} s")
else:
    print("Data does not contain both Single and Multi route stops.")

In [ ]:
# ==========================================
# Correlation Analysis: Route Count vs Delay
# ==========================================

# 1. Group by Route Count and calculate statistics
delay_by_count = base_df_filtered.groupby('route_count_at_stop')['arrival_delay'].agg(['mean', 'count', 'std']).reset_index()
# Filter out route counts with very few samples (e.g., < 100) to avoid noise in the scatter plot
delay_by_count_stable = delay_by_count[delay_by_count['count'] > 100]

# 2. Visualization
plt.figure(figsize=(14, 6))

# Subplot 1: Average Delay vs Route Count
plt.subplot(1, 2, 1)
# Scatter plot weighted by sample size representation (optional, bubble size)
sns.scatterplot(data=delay_by_count_stable, x='route_count_at_stop', y='mean', size='count', sizes=(50, 500), alpha=0.7, legend=False)

# Trend line for ALL data
sns.regplot(data=delay_by_count_stable, x='route_count_at_stop', y='mean', scatter=False, color='red', label='Trend (All)')

# Trend line for Multi-Route Only (>1)
if len(delay_by_count_stable[delay_by_count_stable['route_count_at_stop'] > 1]) > 1:
    sns.regplot(
        data=delay_by_count_stable[delay_by_count_stable['route_count_at_stop'] > 1], 
        x='route_count_at_stop', 
        y='mean', 
        scatter=False, 
        color='green', 
        line_kws={'linestyle': '--'}, # Correct way to pass linestyle
        label='Trend (Excl. 1-route)'
    )

plt.title('Average Delay vs Number of Routes at Stop')
plt.xlabel('Number of Routes at Stop')
plt.ylabel('Average Arrival Delay (s)')
# Legend handling manually since regplot adds to ax but might not label well automatically with sns methods mixed
plt.legend()
plt.grid(True, alpha=0.3)

# Subplot 2: Delay Distribution for common route counts (Boxplot)
plt.subplot(1, 2, 2)
# Dynamically pick top frequent counts to see distribution
top_counts_idx = base_df_filtered['route_count_at_stop'].value_counts().head(8).index.sort_values()
subset_dynamic = base_df_filtered[base_df_filtered['route_count_at_stop'].isin(top_counts_idx)]

sns.boxplot(x='route_count_at_stop', y='arrival_delay', data=subset_dynamic, showfliers=False)
plt.title('Delay Distribution by Route Count (Top Frequent Counts)')
plt.xlabel('Number of Routes at Stop')
plt.ylabel('Arrival Delay (s)')

plt.tight_layout()
plt.show()

# 3. Correlation Coefficients
corr_all = base_df_filtered[['route_count_at_stop', 'arrival_delay']].corr().iloc[0, 1]
print(f"Correlation Coefficient (All Data): {corr_all:.4f}")

subset_multi = base_df_filtered[base_df_filtered['route_count_at_stop'] > 1]
if len(subset_multi) > 0:
    corr_multi = subset_multi[['route_count_at_stop', 'arrival_delay']].corr().iloc[0, 1]
    print(f"Correlation Coefficient (Excluding Single-Route Stops): {corr_multi:.4f}")
else:
    print("No multi-route stops found for second correlation.")

In [ ]:
stop_stats = base_df_filtered.groupby('stop_id').agg({'arrival_delay': 'mean'}).rename(columns={'arrival_delay': 'avg_delay'}).reset_index()

google_types_df = pd.read_csv(const.STOP_GOOGLE_TYPES_CSV)
google_types_df['stop_id'] = google_types_df['stop_id'].astype(str)
stop_stats = stop_stats.merge(google_types_df, on='stop_id', how='left')

In [ ]:
# Install necessary packages
%pip install statsmodels

import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

# 1. データの準備
# ---------------------------------------------------------
# google_types列をリストに変換
google_types_df['types_list'] = google_types_df['google_types'].apply(lambda x: x.split(',') if isinstance(x, str) else [])

# 全ユニークタイプの取得
all_types = set()
for types in google_types_df['types_list']:
    all_types.update(types)

# One-hot encoding
# パフォーマンスのため、辞書リストを作成してからDataFrame化
encoded_data = []
for _, row in google_types_df.iterrows():
    row_data = {'stop_id': row['stop_id']}
    for t in row['types_list']:
        row_data[f'type_{t}'] = 1
    encoded_data.append(row_data)

google_types_encoded = pd.DataFrame(encoded_data).fillna(0)

# stop_idの型変換 (文字列に統一)
stop_stats['stop_id'] = stop_stats['stop_id'].astype(str)
google_types_encoded['stop_id'] = google_types_encoded['stop_id'].astype(str)

# stop_statsとの結合
print("stop_stats columns:", stop_stats.columns)

stop_stats_with_types = pd.merge(stop_stats, google_types_encoded, on='stop_id', how='left')
stop_stats_with_types.fillna(0, inplace=True)

# タイプ列の特定
type_columns = [col for col in stop_stats_with_types.columns if col.startswith('type_')]

# 2. 相関分析
# ---------------------------------------------------------
# 遅延指標
target_col = 'avg_delay'

print(f"Target column for correlation: {target_col}")

if target_col in stop_stats_with_types.columns and len(type_columns) > 0:
    correlations = {}
    for col in type_columns:
        try:
            val = stop_stats_with_types[col]
            if val.nunique() > 1: # 分散がある場合のみ計算
                corr = val.corr(stop_stats_with_types[target_col])
                if not np.isnan(corr):
                    correlations[col] = corr
        except:
            pass

    corr_df = pd.DataFrame(list(correlations.items()), columns=['type', 'correlation'])
    corr_df = corr_df.sort_values(by='correlation', ascending=False)

    print("\nTop 10 Positive Correlations:")
    print(corr_df.head(10))
    print("\nTop 10 Negative Correlations:")
    print(corr_df.tail(10))

# 3. クラスタリング (K-Means)
# ---------------------------------------------------------
if len(type_columns) > 0:
    # クラスタリングに使用するデータ
    X_cluster = stop_stats_with_types[type_columns]
    
    # データを標準化（One Hotなので不要な場合もあるが、クラスタリングでは一般的）
    # ただし今回は0/1なのでそのままでも解釈しやすい
    
    # K-Means実行 (例: 5クラスタ)
    n_clusters = 5
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    stop_stats_with_types['cluster'] = kmeans.fit_predict(X_cluster)

    # クラスタごとの遅延統計
    cluster_stats = stop_stats_with_types.groupby('cluster')[target_col].describe()
    print("\nCluster Statistics (avg_delay):")
    print(cluster_stats)

    # クラスタごとの主な建物タイプの特徴
    cluster_centers = pd.DataFrame(kmeans.cluster_centers_, columns=type_columns)
    print("\nTop feature per cluster (Probability of type presence):")
    for i in range(n_clusters):
        top_features = cluster_centers.iloc[i].sort_values(ascending=False).head(5)
        print(f"Cluster {i}:")
        for ft, val in top_features.items():
            print(f"  {ft}: {val:.2f}")

# 4. 重回帰分析
# ---------------------------------------------------------
if len(type_columns) > 0:
    X = stop_stats_with_types[type_columns].astype(float)
    y = stop_stats_with_types[target_col].astype(float)

    # 定数項の追加
    X = sm.add_constant(X)

    try:
        model = sm.OLS(y, X).fit()
        print("\nRegression Results Summary:")
        # summary() can be large, print key parts or full summary
        print(model.summary())
    except Exception as e:
        print(f"Regression failed: {e}")